In [20]:
import pandas as pd
import importlib
import funciones as func #Tener funciones.py en mismo directorio. Tiene las funciones usadas para procesar un df
importlib.reload(func)
import spotipy
from spotipy.oauth2 import SpotifyOAuth

In [21]:
from generos import *

In [29]:
data = pd.read_csv('../competition_data.csv')
submission = pd.read_csv('../submission.csv')

In [31]:
union = pd.concat([data, submission], axis=0, ignore_index=True)

In [33]:
artistas = (
    union['master_metadata_album_artist_name']  # → esto es una Series
        .dropna()
        .unique()        # devuelve un ndarray con valores únicos
)

In [27]:
artistas = list(artistas)

In [42]:
import webbrowser

In [43]:
auth_manager = SpotifyOAuth(
    client_id="bd28f888d5594351a74597b8c4750b07",
    client_secret="2ad3d9ea7c00425792697d028c5afbd5",
    redirect_uri="http://127.0.0.1:1234/",
    scope="user-top-read",
    open_browser=False
)

# Obtener el URL de autenticación y abrirlo en un navegador
auth_url = auth_manager.get_authorize_url()
print("Por favor, visita este URL para autorizar:", auth_url)
webbrowser.open(auth_url)

# Espera a que el usuario ingrese el código de redirección URL
response = input("Pega el URL completo al que fuiste redirigido aquí: ")
code = auth_manager.parse_response_code(response)
token_info = auth_manager.get_access_token(code)

if token_info:
    print("Token de acceso obtenido:", token_info['access_token'])
else:
    print("No se pudo obtener el token de acceso.")

Por favor, visita este URL para autorizar: https://accounts.spotify.com/authorize?client_id=bd28f888d5594351a74597b8c4750b07&response_type=code&redirect_uri=http%3A%2F%2F127.0.0.1%3A1234%2F&scope=user-top-read


C:\Users\dafyd\AppData\Local\Temp\ipykernel_15580\2729327273.py:17: DeprecationWarning: You're using 'as_dict = True'.get_access_token will return the token string directly in future versions. Please adjust your code accordingly, or use get_cached_token instead.
  token_info = auth_manager.get_access_token(code)


Token de acceso obtenido: BQB52gbmv3VyEQqrVmFIc10hrzo7UBQoY4VNH5EOl6C5qq1rnUpXpi7-okyVW7ItOTg0EBVggTaJkfabiWt4JBANFZLtsdOcJ2P6icG2ivOG10CAxa-tip_D3mhhhWD9IwJLWWoE5rkwClWncWysQp1pOHUxu760DDby6MQrgMIqPinyE_-82wvulvTZyVMX97AM-TEKcVqOoUSperbKSSbUFMPDaL5MC4OZdQ


In [44]:
tracks = union['spotify_track_uri'].dropna().unique()  # Elimina NaN directamente
tracks = [uri for uri in tracks if isinstance(uri, str)]  # Filtra que sean strings

# Dividir en chunks de hasta 50 elementos
chunks = func.chunk_list(tracks, 50)
uri_to_artist = {}
for i, chunk in enumerate(chunks):
    print(i)
    try:
        tracks_info = sp.tracks(chunk)  # Consulta a la API
        for track in tracks_info.get('tracks', []):
            if track:
                uri = track.get('uri')
                artists = track.get('artists')
                uri_to_artist[uri] = artists
            else:
                print("⚠️  Track no encontrado o no disponible.")
    except Exception as e:
        print(f"❌ Error al procesar el chunk {i + 1}: {e}")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231


In [59]:
artist_name_to_id = {}
for clave in uri_to_artist:
    for artista in uri_to_artist[clave]:
        nombre_artista = artista['name']
        id_artista = artista['id']
        if nombre_artista not in artist_name_to_id:
            artist_name_to_id[nombre_artista] = id_artista


In [53]:
artists_ids = list(artist_name_to_id.values())

In [54]:
# Dividir en chunks de hasta 50 elementos
chunks = func.chunk_list(artists_ids, 50)
id_artist_to_genre = {}
for i, chunk in enumerate(chunks):
    print(i)
    try:
        artists_info = sp.artists(chunk)  # Consulta a la API
        for artist in artists_info.get('artists', []):
            if artist:
                aidi = artist.get('id')
                genres = artist.get('genres')
                id_artist_to_genre[aidi] = genres
            else:
                print("⚠️  artista no encontrado o no disponible.")
    except Exception as e:
        print(f"❌ Error al procesar el chunk {i + 1}: {e}")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102


In [55]:
id_artist_to_genre

{'0GWCNkPi54upO9WLlwjAHd': [],
 '7An4yvF7hDYDolN4m5zKBp': ['rock en español',
  'latin rock',
  'argentine rock',
  'latin alternative'],
 '0Xf8oDAJYd2D0k3NLI19OV': ['disco'],
 '0SnyKkoyBaB2fG8IJH4xmU': ['argentine rock'],
 '5UOZXZN4djDSxjfUcpyzzb': [],
 '50INPlL8IzxkVlhNePW4Lz': [],
 '2osoVujXgV0PA8lhqDKYFw': ['argentine rock', 'rock'],
 '2Cd98zHVdZeOCisc6Gi2sB': ['eurodance', 'hip house', 'dance'],
 '53XhwfbYqKCa1cC15pYq2q': [],
 '2ye2Wgw4gimLv2eAKyk1NB': ['metal',
  'thrash metal',
  'rock',
  'heavy metal',
  'hard rock'],
 '1db5TWniHR7iqwXer7AiQ2': ['argentine rock'],
 '73sIBHcqh3Z3NyqHKZ7FOL': [],
 '1bZNv4q3OxYq7mmnLha7Tu': ['argentine rock',
  'latin rock',
  'rock en español',
  'trova'],
 '4Z8W4fKeB5YxbusRsdQVPb': ['art rock', 'alternative rock'],
 '5pf1217gT8zcjOFc7oMi47': [],
 '4nDoRrQiYLoBzwC5BhVJzF': [],
 '1MuQ2m2tg7naeRGAOxYZer': ['argentine rock', 'latin rock', 'rock en español'],
 '1QOmebWGB6FdFtW7Bo3F0W': ['argentine rock',
  'latin rock',
  'rock en español',
  'latin

In [61]:
data['id_artista'] = data['master_metadata_album_artist_name'].map(artist_name_to_id)

In [62]:
data.drop(columns=['1FHygtukKTWaZEEWu0fi2y'], inplace=True)

In [64]:
for i in id_artist_to_genre:
    if len(id_artist_to_genre[i]) > 0:
        id_artist_to_genre[i] = str(id_artist_to_genre[i][0])
    else:
        id_artist_to_genre[i] = 'unknown'

In [66]:
data['artist_genre'] = data['id_artista'].map(id_artist_to_genre)

In [67]:
data

,Unnamed: 0,ts,username,platform,conn_country,user_agent_decrypted,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,reason_start,shuffle,TARGET,id_artista,artist_genre
0,110163,2018-03-11T05:05:45Z,11145402699,"iOS 11.0 (iPhone8,1)",AR,NaN,Crazy For U,Big Time Rush,24/seven,spotify:track:3jFfr89lnSmb4QBtfG8JBP,clickrow,False,False,0GWCNkPi54upO9WLlwjAHd,unknown
1,66026,2023-06-05T10:42:33Z,11145402699,ios,AR,unknown,Nada Personal - Remasterizado 2007,Soda Stereo,Me Verás Volver (Hits & Más),spotify:track:09TTeexnlKewZdjOak2sV2,trackdone,True,False,7An4yvF7hDYDolN4m5zKBp,rock en español
2,116790,2018-07-01T02:04:51Z,11145402699,"iOS 11.0 (iPhone8,1)",AR,unknown,Good Times,CHIC,Risqué,spotify:track:0G3fbPbE1vGeABDEZF0jeG,trackdone,True,True,0Xf8oDAJYd2D0k3NLI19OV,disco
3,18431,2019-09-08T04:58:07Z,11145402699,"iOS 12.4 (iPhone8,1)",AR,unknown,Verano del 92,Los Piojos,3er Arco,spotify:track:1NXvuBAq48QrxRFQZVmORQ,trackdone,False,True,0SnyKkoyBaB2fG8IJH4xmU,argentine rock
4,82941,2017-07-13T18:13:52Z,11145402699,"iOS 11.0 (iPhone8,1)",AR,NaN,Simpatico,Ekko Park,Simpatico,spotify:track:2gYJY0sIx1ErgTIha2nPRg,trackdone,True,False,5UOZXZN4djDSxjfUcpyzzb,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100139,78756,2017-04-06T20:31:14Z,11145402699,"iOS 10.2.1 (iPad6,8,1)",AR,NaN,Losing My Religion,R.E.M.,In Time: The Best Of R.E.M. 1988-2003,spotify:track:12axV6NUqaYH3yFUWwArzr,trackdone,True,False,4KWTAlx2RvbpseOGMEmROg,jangle pop
100140,12585,2021-11-03T21:06:07Z,11145402699,"iOS 15.1 (iPhone12,3)",AR,unknown,Cerca De La Revolucion,Charly García,Piano Bar,spotify:track:66grIvFLGrI4tNhggO2DAd,trackdone,True,False,3jO7X5KupvwmWTHGtHgcgo,argentine rock
100141,93960,2022-01-14T01:13:42Z,11145402699,"iOS 15.2 (iPhone12,3)",AR,unknown,A Los Jóvenes De Ayer - Remastered 2012,Serú Girán,Bicicleta,spotify:track:6YAl320SfxLO4rVIZTFKqG,trackdone,True,False,6CrQKZeuSKNYgrE7PeYqJ1,argentine rock
100142,74339,2024-05-01T17:51:16Z,11145402699,ios,AR,NaN,heat not hot,Serengeti,heat not hot,spotify:track:0rKLO1hXpmoIthQgJKoczN,trackdone,True,False,5F3fDx84RYnmx0FGZeRtSF,experimental hip hop


In [69]:
artist_to_genre = data[['master_metadata_album_artist_name', 'artist_genre']].drop_duplicates()

In [71]:
artist_to_genre.to_csv('artist_to_genre.csv', index=False)